# CoreWeave GPU count estimate

Based on a mix of power capacity and GPU count disclosures from CoreWeave.

Per-year approach:

2023: Geometric interpolation of 10 → 70 MW; all H100/H200.

2024: Fixed 290 MW annual total (70 → 360 MW) split across quarters via a sampled back-loaded ramp; all H100/H200.

2025: Disclosed quarterly MW additions (60/50/120/260) converted to GPUs using the Nvidia sales mix from a lagged anchor quarter (lag = 1q early-2025, shrinking to ~0.4q by Q4 as deployment speeds up).

Q1 2026 pre-purchased in 2025: Share of 850 MW (from capex guidance) scaled by a linear ramp fraction for GPUs bought ahead of power coming online.

In [20]:
import pandas as pd
import numpy as np

# ── Data source ───────────────────────────────────────────────────────────────

FILE_ID = "1tD1uA3SwU86jSYdBxST5VNFkMYyasVrn"
CSV_URL = f"https://drive.google.com/uc?export=download&id={FILE_ID}"

https://drive.google.com/uc?export=download&id=1tD1uA3SwU86jSYdBxST5VNFkMYyasVrn

In [21]:
import pandas as pd
import numpy as np

# ── Data source ───────────────────────────────────────────────────────────────

FILE_ID = "1tD1uA3SwU86jSYdBxST5VNFkMYyasVrn"
CSV_URL = f"https://drive.google.com/uc?export=download&id={FILE_ID}"

# ── Config ────────────────────────────────────────────────────────────────────

N_SIMS = 10000
SEED   = 42

# H100/H200 IT power: fixed point estimate derived from CoreWeave S-1
H100_IT_POWER = 1472  # watts

# B200/B300 IT power multiplier over server power:
# shared uncertainty distribution, centered at 1.14x and slightly tighter
IT_OVER_SERVER_MULT_P5  = 1.08
IT_OVER_SERVER_MULT_P95 = 1.30

B200_SERVER_POWER = 1833  # watts
B300_SERVER_POWER = 1944  # watts

H100E = {"H100/H200": 1.0000, "B200": 2.5265, "B300": 2.5265}
INCLUDE_CHIPS = {"H100/H200", "B200", "B300"}

# 2024 quarterly deployment ramp:
# median quarterly weights for allocating the 2024 annual deployment total.
# We allow uncertainty around these weights, then renormalize so the four
# quarters always sum to 1. This preserves the smooth back-half-weighted ramp
# while allowing some simulations to be flatter or steeper.

RAMP_2024_MEDIANS = {
    "Q1 2024": 0.150,
    "Q2 2024": 0.200,
    "Q3 2024": 0.275,
    "Q4 2024": 0.375,
}

RAMP_2024_P5 = {
    "Q1 2024": 0.080,
    "Q2 2024": 0.130,
    "Q3 2024": 0.190,
    "Q4 2024": 0.270,
}

RAMP_2024_P95 = {
    "Q1 2024": 0.220,
    "Q2 2024": 0.270,
    "Q3 2024": 0.360,
    "Q4 2024": 0.480,
}

Q24_NAMES = ["Q1 2024", "Q2 2024", "Q3 2024", "Q4 2024"]

# ── Lag config per quarter ────────────────────────────────────────────────────
# Q1 2025: fixed at 1q — validated by GB200 NVL72 GA in Feb 2025 aligning with
#          Nvidia first booking B200 revenue in Q4 2024
# Q2 2025: fixed at 1q — validated by GB300 NVL72 deployment in Jul 2025
#          aligning with B300 sales beginning in Q2 2025
# Q3 2025: sampled, median=0.8q — CoreWeave deploying faster as processes mature
# Q4 2025: sampled, median=0.4q — further acceleration; at $30-40k+ per GPU,
#          holding large inventories idle is a significant working capital burden
# Q1 2026 inventory: fixed at 0.4q (median of Q4 distribution) — point estimate
#          for GPUs already purchased but awaiting Q1 2026 power capacity
#
# All sampled lags use sigma=0.4214 (same spread in log space, different medians)
# giving symmetric p5/p95 around the median on a log scale.

LAG_SIGMA = 0.4214

LAG_CONFIG = {
    # quarter: (type, value_or_mu, cap)
    # type="fixed" -> value is the lag in quarters (scalar)
    # type="sampled" -> value is log(median), cap is max lag
    "Q1 2025": ("fixed",   1.0,            None),
    "Q2 2025": ("fixed",   1.0,            None),
    "Q3 2025": ("sampled", np.log(0.8),    2.0),   # P5=0.40q, median=0.80q, P95=1.60q
    "Q4 2025": ("sampled", np.log(0.4),    1.5),   # P5=0.20q, median=0.40q, P95=0.80q
}

# Q1 2026 pre-purchased inventory: fixed at 0.4q
Q1_2026_LAG = 0.4

# Q1 2026 capacity addition
MW_2026_TOTAL = 850
Q1_SHARE_LOW  = 6/35   # ~17.1% from capex guidance ($6B Q1 / $35B full year)
Q1_SHARE_HIGH = 7/30   # ~23.3% from capex guidance ($7B Q1 / $30B full year)

# Nvidia quarters in chronological order
NVIDIA_QUARTERS = [
    "Q4 2022", "Q1 2023", "Q2 2023", "Q3 2023", "Q4 2023",
    "Q1 2024", "Q2 2024", "Q3 2024", "Q4 2024",
    "Q1 2025", "Q2 2025", "Q3 2025", "Q4 2025",
]
Q_INDEX = {q: i for i, q in enumerate(NVIDIA_QUARTERS)}

# Anchor quarters for lag interpolation (lag=1 reference point per CW quarter)
CW_ANCHORS = {
    "Q1 2025": "Q4 2024",
    "Q2 2025": "Q1 2025",
    "Q3 2025": "Q2 2025",
    "Q4 2025": "Q3 2025",
}
CW_MW = {"Q1 2025": 60, "Q2 2025": 50, "Q3 2025": 120, "Q4 2025": 260}

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_URL)
df.columns = df.columns.str.strip().str.lstrip("\ufeff")
df["Start date"] = pd.to_datetime(df["Start date"])
df["End date"]   = pd.to_datetime(df["End date"])
df["Quarter"]    = df["Start date"].dt.to_period("Q").dt.strftime("Q%q %Y")

nvidia = df[
    (df["Chip manufacturer"] == "Nvidia") &
    (df["Chip type"].isin(INCLUDE_CHIPS)) &
    (df["Start date"] >= "2022-10-01") &
    (df["End date"]   <= "2025-12-31")
].copy()

agg = (
    nvidia.groupby(["Quarter", "Chip type"])
    .agg(
        median=("Number of Units", "sum"),
        p5    =("Number of Units (5th percentile)", "sum"),
        p95   =("Number of Units (95th percentile)", "sum"),
    )
    .reset_index()
)

def get_chip_stats(quarter, chip):
    row = agg[(agg["Quarter"] == quarter) & (agg["Chip type"] == chip)]
    if row.empty:
        return 0, 0, 0
    return row["median"].values[0], row["p5"].values[0], row["p95"].values[0]

# ── Lognormal helpers ─────────────────────────────────────────────────────────

Z95 = 1.6449

def lognormal_params(p5, p95):
    sigma = (np.log(p95) - np.log(p5)) / (2 * Z95)
    mu    = (np.log(p95) + np.log(p5)) / 2
    return mu, sigma

def sample_units(p5, p95, n, rng):
    if p5 == 0 and p95 == 0:
        return np.zeros(n)
    mu, sigma = lognormal_params(p5, p95)
    return rng.lognormal(mu, sigma, n)

# ── Sample all random quantities ──────────────────────────────────────────────

rng = np.random.default_rng(SEED)

# 2024 quarterly ramp weights: sample each quarter's raw weight around its
# target range, then renormalize so all four sum to 1.
q24_weight_draws_raw = {}
for q in Q24_NAMES:
    mu_q, sigma_q = lognormal_params(RAMP_2024_P5[q], RAMP_2024_P95[q])
    q24_weight_draws_raw[q] = rng.lognormal(mu_q, sigma_q, N_SIMS)

q24_weight_sum = np.zeros(N_SIMS)
for q in Q24_NAMES:
    q24_weight_sum += q24_weight_draws_raw[q]

q24_weight_draws = {
    q: q24_weight_draws_raw[q] / q24_weight_sum
    for q in Q24_NAMES
}

# Per-quarter lag samples (fixed or sampled independently)
lag_draws = {}
for quarter, (lag_type, value, cap) in LAG_CONFIG.items():
    if lag_type == "fixed":
        lag_draws[quarter] = np.full(N_SIMS, value)
    else:
        samples = rng.lognormal(value, LAG_SIGMA, N_SIMS)
        lag_draws[quarter] = np.minimum(samples, cap)

# Q1 2026 inventory lag: fixed
lag_draws["Q1 2026 (pre-purchased)"] = np.full(N_SIMS, Q1_2026_LAG)

# Shared draw for B200/B300 IT-over-server multiplier
z_efficiency = rng.standard_normal(N_SIMS)
mu_mult, sigma_mult = lognormal_params(IT_OVER_SERVER_MULT_P5, IT_OVER_SERVER_MULT_P95)
it_over_server_mult = np.exp(mu_mult + sigma_mult * z_efficiency)

b200_it_power = B200_SERVER_POWER * it_over_server_mult
b300_it_power = B300_SERVER_POWER * it_over_server_mult

# Q1 2026 capex share: uniform
q1_2026_share = rng.uniform(Q1_SHARE_LOW, Q1_SHARE_HIGH, N_SIMS)
q1_2026_mw_total = MW_2026_TOTAL * q1_2026_share

# Linear ramp adjustment: capacity comes online gradually throughout Q1 2026,
# not all at once on Jan 1. With a lag of L quarters and a linear ramp over the
# quarter, CoreWeave has only pre-purchased the fraction (L / 1 quarter) of Q1's
# total MW addition by end of Q4 2025. Capped at 1.0 (can't pre-purchase more
# than the full quarter's addition).
# With fixed lag of 0.4q: fraction = 0.4, so only 40% of Q1 2026 MW is pre-purchased.
q1_2026_ramp_fraction = np.minimum(lag_draws["Q1 2026 (pre-purchased)"] / 1.0, 1.0)
q1_2026_mw = q1_2026_mw_total * q1_2026_ramp_fraction

# Nvidia unit counts
nvidia_samples = {}
for q in NVIDIA_QUARTERS:
    for chip in ["H100/H200", "B200", "B300"]:
        _, p5, p95 = get_chip_stats(q, chip)
        nvidia_samples[(q, chip)] = sample_units(p5, p95, N_SIMS, rng)

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_it_power(chip):
    if chip == "H100/H200":
        return H100_IT_POWER
    if chip == "B200":
        return b200_it_power
    if chip == "B300":
        return b300_it_power
    raise ValueError(f"Unknown chip: {chip}")

def get_interpolated_shares(anchor_q, lag):
    anchor_idx = Q_INDEX[anchor_q]
    float_idx  = np.clip(anchor_idx - (lag - 1), 0, len(NVIDIA_QUARTERS) - 1)
    idx_low    = np.floor(float_idx).astype(int)
    idx_high   = np.ceil(float_idx).astype(int)
    w_high     = float_idx - idx_low
    w_low      = 1.0 - w_high

    total_it = np.zeros(N_SIMS)
    interp   = {}
    for chip in ["H100/H200", "B200", "B300"]:
        ul = np.array([nvidia_samples[(NVIDIA_QUARTERS[i], chip)][s]
                       for s, i in enumerate(idx_low)])
        uh = np.array([nvidia_samples[(NVIDIA_QUARTERS[i], chip)][s]
                       for s, i in enumerate(idx_high)])
        interp[chip] = w_low * ul + w_high * uh
        total_it    += interp[chip] * get_it_power(chip)

    return {
        chip: np.where(total_it > 0, interp[chip] * get_it_power(chip) / total_it, 0.0)
        for chip in interp
    }

def compute_gpu_counts(mw_added, shares):
    units = {chip: mw_added * shares[chip] * 1e6 / get_it_power(chip) for chip in shares}
    h100e = sum(units[chip] * H100E[chip] for chip in units)
    total = sum(units[chip] for chip in units)
    return units, total, h100e

# ── Monte Carlo ───────────────────────────────────────────────────────────────

METRICS = ["H100/H200", "B200", "B300", "Total", "H100e"]
sim_results = {}

# 2025 deployed quarters
for cw_quarter, mw_added in CW_MW.items():
    shares = get_interpolated_shares(CW_ANCHORS[cw_quarter], lag_draws[cw_quarter])
    units, total, h100e = compute_gpu_counts(mw_added, shares)
    sim_results[cw_quarter] = {
        "H100/H200": units["H100/H200"],
        "B200": units["B200"],
        "B300": units["B300"],
        "Total": total,
        "H100e": h100e,
    }

# Q1 2026 pre-purchased
q1_2026_shares = get_interpolated_shares("Q4 2025", lag_draws["Q1 2026 (pre-purchased)"])
q1_units, q1_total, q1_h100e = compute_gpu_counts(q1_2026_mw, q1_2026_shares)
sim_results["Q1 2026 (pre-purchased)"] = {
    "H100/H200": q1_units["H100/H200"],
    "B200": q1_units["B200"],
    "B300": q1_units["B300"],
    "Total": q1_total,
    "H100e": q1_h100e,
}

# ── 2023 geometric quarterly interpolation ───────────────────────────────────
# CW data-center capacity: 10 MW end-2022 → 70 MW end-2023 → 360 MW end-2024.
# Geometric interpolation: constant quarterly growth factor = R^(1/4), so that
# end-of-Qk capacity = C0 * R^(k/4).  This avoids the dramatic under/overstatement
# of early/late quarters that linear interpolation produces for 5-7× annual growth.

MW_END_2022 = 10.0
MW_END_2023 = 70.0
MW_END_2024 = 360.0

R_2023       = MW_END_2023 / MW_END_2022        # = 7.0
Q_FACTOR_2023 = R_2023 ** 0.25                  # ≈ 1.6266 (≈63%/qtr)

# End-of-quarter MW levels
_mw_eoq = {
    "Q4 2022": MW_END_2022,
    "Q1 2023": MW_END_2022 * Q_FACTOR_2023,
    "Q2 2023": MW_END_2022 * Q_FACTOR_2023 ** 2,
    "Q3 2023": MW_END_2022 * Q_FACTOR_2023 ** 3,
    "Q4 2023": MW_END_2023,     # pinned to observed anchor
}

# MW additions per quarter (differences of end-of-quarter levels)
MW_ADD_2023 = {
    "Q1 2023": _mw_eoq["Q1 2023"] - _mw_eoq["Q4 2022"],
    "Q2 2023": _mw_eoq["Q2 2023"] - _mw_eoq["Q1 2023"],
    "Q3 2023": _mw_eoq["Q3 2023"] - _mw_eoq["Q2 2023"],
    "Q4 2023": _mw_eoq["Q4 2023"] - _mw_eoq["Q3 2023"],
}

Q23_NAMES = ["Q1 2023", "Q2 2023", "Q3 2023", "Q4 2023"]

# GPU count basis: 65 MW, not 60 MW.
# Net capacity increased 60 MW (10→70 MW), but ~5 MW was freed by retiring 2022
# chips, so gross capacity used by new 2023 GPUs ≈ 65 MW — consistent with
# 65 MW / 1,472 W ≈ 44,158 GPUs × 1,472 W = 65 MW and the S-1 fleet total of ~250k.
# The geometric *shape* is still derived from 10→70 MW capacity anchors;
# quarterly GPU counts are the geometric shares scaled to 65 MW total.
point_2023_total_mw = MW_END_2023 - MW_END_2022          # = 60 MW net capacity gain
MW_2023_GPU_BASIS   = 65.0  # MW basis for GPU counting (accounts for ~5 MW retirements)
point_2023_gpus = round(MW_2023_GPU_BASIS * 1e6 / H100_IT_POWER)

point_2024_total_gpus = round((MW_END_2024 - MW_END_2023) * 1e6 / H100_IT_POWER)

# ── Point estimates and sampled quarterly arrays ──────────────────────────────

# 2023 quarterly arrays: geometric point estimates (all H100/H200, no uncertainty)
# Scale quarterly GPU counts: geometric shape (from MW_ADD_2023) × 65 MW basis
_mw_add_2023_sum = sum(MW_ADD_2023.values())  # = 60.0 MW
q23_arrays = {}
for q in Q23_NAMES:
    _gpus23 = round(MW_ADD_2023[q] / _mw_add_2023_sum * MW_2023_GPU_BASIS * 1e6 / H100_IT_POWER)
    q23_arrays[q] = {
        "H100/H200": np.full(N_SIMS, _gpus23),
        "B200":      np.zeros(N_SIMS),
        "B300":      np.zeros(N_SIMS),
        "Total":     np.full(N_SIMS, _gpus23),
        "H100e":     np.full(N_SIMS, _gpus23),
    }

# 2024 quarterly arrays: sampled weight × fixed 2024 annual total
q24_arrays = {}
for q in Q24_NAMES:
    q24_total = point_2024_total_gpus * q24_weight_draws[q]
    q24_arrays[q] = {
        "H100/H200": q24_total,
        "B200": np.zeros(N_SIMS),
        "B300": np.zeros(N_SIMS),
        "Total": q24_total,
        "H100e": q24_total,
    }

# ── Summarise ─────────────────────────────────────────────────────────────────

def pct(arr, q):
    return int(round(np.percentile(arr, q)))

# 2023 annual total arrays
q23_total_arrays = {m: np.zeros(N_SIMS) for m in METRICS}
for q in Q23_NAMES:
    for m in METRICS:
        q23_total_arrays[m] += q23_arrays[q][m]

# 2024 annual total arrays
q24_total_arrays = {m: np.zeros(N_SIMS) for m in METRICS}
for q in Q24_NAMES:
    for m in METRICS:
        q24_total_arrays[m] += q24_arrays[q][m]

# 2025 deployed subtotal
q25_arrays = {m: np.zeros(N_SIMS) for m in METRICS}
for q in CW_MW:
    for m in METRICS:
        q25_arrays[m] += sim_results[q][m]

# End-of-2025 inventory = 2025 deployed + Q1 2026 pre-purchased
inv_arrays = {m: q25_arrays[m] + sim_results["Q1 2026 (pre-purchased)"][m] for m in METRICS}

# Grand totals
grand_arrays = {
    m: inv_arrays[m] + q24_total_arrays[m] + q23_total_arrays[m]
    for m in METRICS
}

# ── Print ─────────────────────────────────────────────────────────────────────

W = 80
HDR = f"{'Metric':40s} {'P5':>10s} {'Median':>10s} {'P95':>10s}"
SEP = "-" * 70

def print_point(title, rows_dict):
    print(f"\n{'='*W}\n  {title} (point estimate)\n{'='*W}")
    print(HDR)
    print(SEP)
    for m in METRICS:
        v = rows_dict[m]
        print(f"  {m:38s} {v:>10,} {v:>10,} {v:>10,}")

def print_mc(title, quarters=None, arrays=None, source_dict=None):
    print(f"\n{'='*W}\n  {title}\n{'='*W}")
    print(HDR)
    print(SEP)
    if quarters:
        for q in quarters:
            print(f"\n  {q}")
            for m in METRICS:
                arr = source_dict[q][m]
                print(f"    {m:36s} {pct(arr,5):>10,} {pct(arr,50):>10,} {pct(arr,95):>10,}")
    if arrays is not None:
        for m in METRICS:
            print(f"  {m:38s} {pct(arrays[m],5):>10,} {pct(arrays[m],50):>10,} {pct(arrays[m],95):>10,}")

print_mc("2023 QUARTERLY DEPLOYMENTS (geometric interpolation — point estimate)",
         quarters=Q23_NAMES, source_dict=q23_arrays)
print_mc("2023 DEPLOYED TOTAL (point estimate)", arrays=q23_total_arrays)

print_mc("2024 QUARTERLY DEPLOYMENTS (smooth ramp with uncertainty)", quarters=Q24_NAMES, source_dict=q24_arrays)
print_mc("2024 DEPLOYED TOTAL", arrays=q24_total_arrays)

print_mc("2025 QUARTERLY DEPLOYMENTS", quarters=list(CW_MW.keys()), source_dict=sim_results)
print_mc("Q1 2026 PRE-PURCHASED INVENTORY (held at end of 2025)",
         quarters=["Q1 2026 (pre-purchased)"], source_dict=sim_results)
print_mc("2025 DEPLOYED TOTAL", arrays=q25_arrays)
print_mc("END-OF-2025 INVENTORY (deployed + pre-purchased)", arrays=inv_arrays)
print_mc("GRAND TOTAL incl. 2023/2024 (deployed + pre-purchased)", arrays=grand_arrays)

# ── Validation ────────────────────────────────────────────────────────────────

print(f"\n{'='*W}\n  VALIDATION\n{'='*W}")
fleet_end_2024 = round(17000 / 2) + point_2023_gpus + point_2024_total_gpus
print(f"\nFleet end-2024: {fleet_end_2024:,}  (S-1 disclosed: 250,000)")
print(f"  [65 MW GPU basis (60 MW net + ~5 MW freed from 2022 retirements).]")
print(f"IT power/GPU 2023: {MW_2023_GPU_BASIS*1e6/point_2023_gpus:.0f} W  (anchor: 1,472 W)")
print(f"IT power/GPU 2024: {(MW_END_2024-MW_END_2023)*1e6/point_2024_total_gpus:.0f} W  (anchor: 1,472 W)")

print(f"\n2023 geometric quarterly GPU additions (R=7, factor={Q_FACTOR_2023:.4f}/qtr, 65 MW basis):")
for q in Q23_NAMES:
    gpus_q = round(MW_ADD_2023[q] / _mw_add_2023_sum * MW_2023_GPU_BASIS * 1e6 / H100_IT_POWER)
    share  = MW_ADD_2023[q] / _mw_add_2023_sum * 100
    print(f"  {q}: {MW_ADD_2023[q]:.2f} MW capacity shape  ({share:.1f}% of year)  → {gpus_q:,} GPUs")

print(f"\n2024 ramp weights:")
for q in Q24_NAMES:
    print(f"  {q}: P5={np.percentile(q24_weight_draws[q],5)*100:.1f}%, "
          f"Median={np.percentile(q24_weight_draws[q],50)*100:.1f}%, "
          f"P95={np.percentile(q24_weight_draws[q],95)*100:.1f}%")

print(f"\nLag summary:")
for q, (lag_type, value, cap) in LAG_CONFIG.items():
    if lag_type == "fixed":
        print(f"  {q}: fixed={value}q")
    else:
        arr = lag_draws[q]
        print(
            f"  {q}: P5={np.percentile(arr,5):.2f}q, Median={np.percentile(arr,50):.2f}q, "
            f"P95={np.percentile(arr,95):.2f}q, cap={cap}q, "
            f"sims capped={np.mean(arr==cap)*100:.2f}%"
        )
print(f"  Q1 2026 inventory: fixed={Q1_2026_LAG}q")

print(f"\nShared IT/server multiplier: P5={np.percentile(it_over_server_mult,5):.3f}, "
      f"Median={np.percentile(it_over_server_mult,50):.3f}, "
      f"P95={np.percentile(it_over_server_mult,95):.3f}")
print(f"B200 IT power: P5={np.percentile(b200_it_power,5):,.2f} W, "
      f"Median={np.percentile(b200_it_power,50):,.2f} W, "
      f"P95={np.percentile(b200_it_power,95):,.2f} W")
print(f"B300 IT power: P5={np.percentile(b300_it_power,5):,.2f} W, "
      f"Median={np.percentile(b300_it_power,50):,.2f} W, "
      f"P95={np.percentile(b300_it_power,95):,.2f} W")
print(f"Q1 2026 total MW addition: P5={pct(q1_2026_mw_total,5):,}, Median={pct(q1_2026_mw_total,50):,}, P95={pct(q1_2026_mw_total,95):,} MW")
print(f"Q1 2026 pre-purchased MW (ramp adj, lag={Q1_2026_LAG}q -> {Q1_2026_LAG*100:.0f}% of quarter): "
      f"P5={pct(q1_2026_mw,5):,}, Median={pct(q1_2026_mw,50):,}, P95={pct(q1_2026_mw,95):,} MW")

print(f"\n{'='*W}\n  UNCERTAINTY SOURCES MODELLED\n{'='*W}")
print("  (1) Nvidia unit counts: lognormal fitted to p5/p95 per chip per quarter.")
print("  (2) Deployment lag: per-quarter, independently sampled where not fixed.")
print("      Q1/Q2 2025: fixed at 1q (validated by deployment disclosures).")
print("      Q3 2025: lognormal, P5=0.40q, median=0.80q, P95=1.60q, cap=2.0q.")
print("      Q4 2025: lognormal, P5=0.20q, median=0.40q, P95=0.80q, cap=1.5q.")
print("      Q1 2026 inventory: fixed at 0.4q.")
print("  (3) B200/B300 IT power: shared lognormal multiplier over server power.")
print("      Shared multiplier: p5=1.08x, median=1.14x, p95=1.20x.")
print(f"      B200 IT power = {B200_SERVER_POWER:,} W × shared multiplier.")
print(f"      B300 IT power = {B300_SERVER_POWER:,} W × shared multiplier.")
print("  (4) Q1 2026 capex share: uniform(17.1%, 23.3%).")
print("  (5) 2024 quarterly deployment path is inferred from a smooth ramp with uncertainty.")
print("      Quarterly weights are sampled around medians of 15.0%, 20.0%, 27.5%, 37.5%,")
print("      using wider raw ranges, then renormalized to sum to 100% of the 2024 annual deployment total.")
print("  H100/H200 IT power fixed at 1,472 W.")
print("  2023: geometric quarterly interpolation (R=7, factor≈1.627/qtr).")
print("        Point estimates only — no Monte Carlo uncertainty on 2023 quarterly split.")


  2023 QUARTERLY DEPLOYMENTS (geometric interpolation — point estimate)
Metric                                           P5     Median        P95
----------------------------------------------------------------------

  Q1 2023
    H100/H200                                 4,611      4,611      4,611
    B200                                          0          0          0
    B300                                          0          0          0
    Total                                     4,611      4,611      4,611
    H100e                                     4,611      4,611      4,611

  Q2 2023
    H100/H200                                 7,501      7,501      7,501
    B200                                          0          0          0
    B300                                          0          0          0
    Total                                     7,501      7,501      7,501
    H100e                                     7,501      7,501      7,501

  Q3 2023
    H100/

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Airtable output: quarters_by_chip
#
# Produces one row per chip per quarter (not cumulative).
# Q4 2025 folds in Q1 2026 pre-purchased inventory — no separate label.
# TDP uses GPU chip power: H100/H200=700W, B200=1200W, B300=1400W.
# Skip rows where both median and p95 are 0.
# Columns match owners_csv_export/nvidia_owners_quarters_by_chip.csv schema.
# ══════════════════════════════════════════════════════════════════════════════

GPU_POWER_W = {"H100/H200": 700, "B200": 1200, "B300": 1400}

QUARTER_DATES = {
    "Q1 2023": ("1/1/2023",  "3/31/2023"),
    "Q2 2023": ("4/1/2023",  "6/30/2023"),
    "Q3 2023": ("7/1/2023",  "9/30/2023"),
    "Q4 2023": ("10/1/2023", "12/31/2023"),
    "Q1 2024": ("1/1/2024",  "3/31/2024"),
    "Q2 2024": ("4/1/2024",  "6/30/2024"),
    "Q3 2024": ("7/1/2024",  "9/30/2024"),
    "Q4 2024": ("10/1/2024", "12/31/2024"),
    "Q1 2025": ("1/1/2025",  "3/31/2025"),
    "Q2 2025": ("4/1/2025",  "6/30/2025"),
    "Q3 2025": ("7/1/2025",  "9/30/2025"),
    "Q4 2025": ("10/1/2025", "12/31/2025"),
}

GENERATED_NOTE = f"Estimates generated on: {pd.Timestamp.now().strftime('%m-%d-%Y %H:%M')}"

Q4_2025_NOTE = (
    "Includes Q1 2026 pre-purchased inventory (GPUs held but not yet deployed); "
    "prior quarters count deployed chips only."
)

def make_row_by_chip(name, chip, start, end, arr, extra_note=None):
    med = pct(arr, 50)
    p5  = pct(arr, 5)
    p95 = pct(arr, 95)
    pw  = GPU_POWER_W[chip]
    notes = f"{extra_note} {GENERATED_NOTE}" if extra_note else GENERATED_NOTE
    return {
        "Name":                                name,
        "Chip manufacturer":                   "Nvidia",
        "Owner":                               "CoreWeave",
        "Start date":                          start,
        "End date":                            end,
        "Compute estimate in H100e (median)":  int(round(med * H100E[chip])),
        "H100e (5th percentile)":              int(round(p5  * H100E[chip])),
        "H100e (95th percentile)":             int(round(p95 * H100E[chip])),
        "Number of Units":                     med,
        "Number of Units (5th percentile)":    p5,
        "Number of Units (95th percentile)":   p95,
        "Total TDP (W)":                       med * pw,
        "Total TDP (W) (5th percentile)":      p5  * pw,
        "Total TDP (W) (95th percentile)":     p95 * pw,
        "Chip type":                           chip,
        "Incomplete":                          None,
        "Source / Link":                       None,
        "Notes":                               notes,
    }

rows_qbychip = []

# ── 2023 Q1–Q4: all H100/H200 (geometric point estimates) ──────────────────────
for q in Q23_NAMES:
    start, end = QUARTER_DATES[q]
    rows_qbychip.append(make_row_by_chip(
        name=f"CoreWeave H100/H200 {q}",
        chip="H100/H200", start=start, end=end,
        arr=q23_arrays[q]["H100/H200"],
    ))

# ── 2024 Q1–Q4: all H100/H200 ─────────────────────────────────────────────────
for q in Q24_NAMES:
    start, end = QUARTER_DATES[q]
    rows_qbychip.append(make_row_by_chip(
        name=f"CoreWeave H100/H200 {q}",
        chip="H100/H200", start=start, end=end,
        arr=q24_arrays[q]["H100/H200"],
    ))

# ── 2025 Q1–Q3: all three chip types ──────────────────────────────────────────
for q in ["Q1 2025", "Q2 2025", "Q3 2025"]:
    start, end = QUARTER_DATES[q]
    for chip in ["H100/H200", "B200", "B300"]:
        arr = sim_results[q][chip]
        if pct(arr, 50) == 0 and pct(arr, 95) == 0:
            continue
        rows_qbychip.append(make_row_by_chip(
            name=f"CoreWeave {chip} {q}",
            chip=chip, start=start, end=end, arr=arr,
        ))

# ── Q4 2025: deployed + pre-purchased inventory folded in ─────────────────────
start, end = QUARTER_DATES["Q4 2025"]
for chip in ["H100/H200", "B200", "B300"]:
    arr = sim_results["Q4 2025"][chip] + sim_results["Q1 2026 (pre-purchased)"][chip]
    if pct(arr, 50) == 0 and pct(arr, 95) == 0:
        continue
    rows_qbychip.append(make_row_by_chip(
        name=f"CoreWeave {chip} Q4 2025",
        chip=chip, start=start, end=end, arr=arr,
        extra_note=Q4_2025_NOTE,
    ))

df_quarters_by_chip = pd.DataFrame(rows_qbychip)
print(f"quarters_by_chip: {len(df_quarters_by_chip)} rows × {len(df_quarters_by_chip.columns)} cols\n")
print(df_quarters_by_chip.to_string(index=False))

quarters_by_chip: 18 rows × 18 cols

                       Name Chip manufacturer     Owner Start date   End date  Compute estimate in H100e (median)  H100e (5th percentile)  H100e (95th percentile)  Number of Units  Number of Units (5th percentile)  Number of Units (95th percentile)  Total TDP (W)  Total TDP (W) (5th percentile)  Total TDP (W) (95th percentile) Chip type Incomplete Source / Link                                                                                                                                                         Notes
CoreWeave H100/H200 Q1 2023            Nvidia CoreWeave   1/1/2023  3/31/2023                                4611                    4611                     4611             4611                              4611                               4611        3227700                         3227700                          3227700 H100/H200       None          None                                                                              

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Airtable output: cumulative_by_chip
#
# Cumulative per chip per quarter, starting Q1 2023.
# 2023 quarterly deployments are geometric point estimates; 2024+ use Monte Carlo.
# Q4 2025 includes deployed + pre-purchased inventory.
# Depends on: GPU_POWER_W, QUARTER_DATES, GENERATED_NOTE, make_row_by_chip
#             (all defined in Cell 1 above).
# ══════════════════════════════════════════════════════════════════════════════

CUMUL_START = "1/1/2023"  # expanded: 2023 now shown quarterly

CUMUL_END_DATES = {
    "Q1 2023": "3/31/2023",  "Q2 2023": "6/30/2023",
    "Q3 2023": "9/30/2023",  "Q4 2023": "12/31/2023",
    "Q1 2024": "3/31/2024",  "Q2 2024": "6/30/2024",
    "Q3 2024": "9/30/2024",  "Q4 2024": "12/31/2024",
    "Q1 2025": "3/31/2025",  "Q2 2025": "6/30/2025",
    "Q3 2025": "9/30/2025",  "Q4 2025": "12/31/2025",
}

ALL_QUARTERS = Q23_NAMES + Q24_NAMES + list(CW_MW.keys())  # Q1 2023 … Q4 2025

rows_cumbychip = []
cumul = {chip: np.zeros(N_SIMS) for chip in ["H100/H200", "B200", "B300"]}
# 2023 is now iterated quarter-by-quarter; no pre-loaded starting balance

for q in ALL_QUARTERS:
    # Add this quarter's deployments
    if q in Q23_NAMES:
        cumul["H100/H200"] += q23_arrays[q]["H100/H200"]
    elif q in Q24_NAMES:
        cumul["H100/H200"] += q24_arrays[q]["H100/H200"]
    elif q == "Q4 2025":
        for chip in ["H100/H200", "B200", "B300"]:
            cumul[chip] += sim_results[q][chip] + sim_results["Q1 2026 (pre-purchased)"][chip]
    else:
        for chip in ["H100/H200", "B200", "B300"]:
            cumul[chip] += sim_results[q][chip]

    end = CUMUL_END_DATES[q]
    for chip in ["H100/H200", "B200", "B300"]:
        arr = cumul[chip]
        if pct(arr, 50) == 0 and pct(arr, 95) == 0:
            continue
        rows_cumbychip.append(make_row_by_chip(
            name=f"CoreWeave {chip} cumulative through {q}",
            chip=chip, start=CUMUL_START, end=end, arr=arr,
        ))

df_cumulative_by_chip = pd.DataFrame(rows_cumbychip)
print(f"cumulative_by_chip: {len(df_cumulative_by_chip)} rows × {len(df_cumulative_by_chip.columns)} cols\n")
print(df_cumulative_by_chip.to_string(index=False))


cumulative_by_chip: 18 rows × 18 cols

                                          Name Chip manufacturer     Owner Start date   End date  Compute estimate in H100e (median)  H100e (5th percentile)  H100e (95th percentile)  Number of Units  Number of Units (5th percentile)  Number of Units (95th percentile)  Total TDP (W)  Total TDP (W) (5th percentile)  Total TDP (W) (95th percentile) Chip type Incomplete Source / Link                                    Notes
CoreWeave H100/H200 cumulative through Q1 2023            Nvidia CoreWeave   1/1/2023  3/31/2023                                4611                    4611                     4611             4611                              4611                               4611        3227700                         3227700                          3227700 H100/H200       None          None Estimates generated on: 04-22-2026 12:04
CoreWeave H100/H200 cumulative through Q2 2023            Nvidia CoreWeave   1/1/2023  6/30/2023                 

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Airtable output: cumulative_totals
#
# Cumulative across all chip types per quarter (no Chip type column).
# TDP is a distribution-level sum: each sim draw weights each chip correctly,
# so the percentiles reflect true uncertainty in the chip mix.
# Starting Q1 2023; 2023 quarterly values are geometric point estimates.
# Columns match owners_csv_export/nvidia_owners_cumulative_totals.csv schema.
# Depends on: GPU_POWER_W, CUMUL_START, CUMUL_END_DATES, ALL_QUARTERS,
#             GENERATED_NOTE (all defined in Cells 1–2 above).
# ══════════════════════════════════════════════════════════════════════════════

def make_row_totals(name, start, end, units_arr, h100e_arr, tdp_arr):
    return {
        "Name":                                name,
        "Chip manufacturer":                   "Nvidia",
        "Owner":                               "CoreWeave",
        "Start date":                          start,
        "End date":                            end,
        "Compute estimate in H100e (median)":  pct(h100e_arr, 50),
        "H100e (5th percentile)":              pct(h100e_arr, 5),
        "H100e (95th percentile)":             pct(h100e_arr, 95),
        "Number of Units":                     pct(units_arr, 50),
        "Number of Units (5th percentile)":    pct(units_arr, 5),
        "Number of Units (95th percentile)":   pct(units_arr, 95),
        "Total TDP (W)":                       pct(tdp_arr, 50),
        "Total TDP (W) (5th percentile)":      pct(tdp_arr, 5),
        "Total TDP (W) (95th percentile)":     pct(tdp_arr, 95),
        "Power in MW (median)":                round(pct(tdp_arr, 50) / 1e6, 2),
        "Power in MW (5th percentile)":        round(pct(tdp_arr, 5)  / 1e6, 2),
        "Power in MW (95th percentile)":       round(pct(tdp_arr, 95) / 1e6, 2),
        "Incomplete":                          None,
        "Source / Link":                       None,
        "Notes":                               GENERATED_NOTE,
    }

rows_cumtotals = []
cumul_t = {chip: np.zeros(N_SIMS) for chip in ["H100/H200", "B200", "B300"]}
# 2023 is now iterated quarter-by-quarter; no pre-loaded starting balance

for q in ALL_QUARTERS:
    if q in Q23_NAMES:
        cumul_t["H100/H200"] += q23_arrays[q]["H100/H200"]
    elif q in Q24_NAMES:
        cumul_t["H100/H200"] += q24_arrays[q]["H100/H200"]
    elif q == "Q4 2025":
        for chip in ["H100/H200", "B200", "B300"]:
            cumul_t[chip] += sim_results[q][chip] + sim_results["Q1 2026 (pre-purchased)"][chip]
    else:
        for chip in ["H100/H200", "B200", "B300"]:
            cumul_t[chip] += sim_results[q][chip]

    # Aggregate across chips at the simulation-draw level (preserves correlation)
    units_arr = cumul_t["H100/H200"] + cumul_t["B200"] + cumul_t["B300"]
    h100e_arr = (cumul_t["H100/H200"] * H100E["H100/H200"]
               + cumul_t["B200"]      * H100E["B200"]
               + cumul_t["B300"]      * H100E["B300"])
    tdp_arr   = (cumul_t["H100/H200"] * GPU_POWER_W["H100/H200"]
               + cumul_t["B200"]      * GPU_POWER_W["B200"]
               + cumul_t["B300"]      * GPU_POWER_W["B300"])

    rows_cumtotals.append(make_row_totals(
        name=f"CoreWeave cumulative through {q}",
        start=CUMUL_START, end=CUMUL_END_DATES[q],
        units_arr=units_arr, h100e_arr=h100e_arr, tdp_arr=tdp_arr,
    ))

df_cumulative_totals = pd.DataFrame(rows_cumtotals)
print(f"cumulative_totals: {len(df_cumulative_totals)} rows × {len(df_cumulative_totals.columns)} cols\n")
print(df_cumulative_totals.to_string(index=False))

cumulative_totals: 12 rows × 20 cols

                                Name Chip manufacturer     Owner Start date   End date  Compute estimate in H100e (median)  H100e (5th percentile)  H100e (95th percentile)  Number of Units  Number of Units (5th percentile)  Number of Units (95th percentile)  Total TDP (W)  Total TDP (W) (5th percentile)  Total TDP (W) (95th percentile)  Power in MW (median)  Power in MW (5th percentile)  Power in MW (95th percentile) Incomplete Source / Link                                    Notes
CoreWeave cumulative through Q1 2023            Nvidia CoreWeave   1/1/2023  3/31/2023                                4611                    4611                     4611             4611                              4611                               4611        3227700                         3227700                          3227700                  3.23                          3.23                           3.23       None          None Estimates generated on: 04-22

In [25]:
import os

OUT_DIR = "owners_csv_export/coreweave"
os.makedirs(OUT_DIR, exist_ok=True)

df_quarters_by_chip.to_csv(f"{OUT_DIR}/coreweave_owners_quarters_by_chip.csv", index=False)
df_cumulative_by_chip.to_csv(f"{OUT_DIR}/coreweave_owners_cumulative_by_chip.csv", index=False)
df_cumulative_totals.to_csv(f"{OUT_DIR}/coreweave_owners_cumulative_totals.csv", index=False)

print(f"Wrote 3 CSVs to {OUT_DIR}/")

Wrote 3 CSVs to owners_csv_export/coreweave/
